In [ ]:
# Cell 1: Imports and Setup
# همه importها رو اینجا بذار تا بقیه سلول‌ها وابسته نباشن
from datasets import DatasetDict, Dataset
import torch
import os
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, BitsAndBytesConfig, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModelForSeq2SeqLM
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import re
import evaluate
import math

print("✅ Imports done!")


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports done!


In [ ]:
# Cell 2: Load Dataset
validated_split = Dataset.load_from_disk("/data/whisper/dataset/CV-17-01/validated")

print(validated_split.select(range(1)))
print(validated_split.select(range(1))['audio'])

# Split into train/validation
dataset_dict = validated_split.train_test_split(
    test_size=0.2,    # 20% validation
    seed=42,          # for reproducibility
    shuffle=True
)

# Rename 'test' to 'validation' for clarity
dataset_dict = DatasetDict({
    "train": dataset_dict["train"],
    "validation": dataset_dict["test"]
})

print(dataset_dict["train"].select(range(1)))
print(dataset_dict["train"].select(range(1))['audio'])
print(dataset_dict["validation"].select(range(1)))
print(dataset_dict["validation"].select(range(1))['audio'])

print("✅ Dataset loaded and split!")


Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant'],
    num_rows: 1
})
[{'path': None, 'array': array([-9.15527344e-05, -9.15527344e-05, -1.52587891e-04, ...,
        5.79833984e-04,  4.88281250e-04,  5.79833984e-04], shape=(42752,)), 'sampling_rate': 16000}]
Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant'],
    num_rows: 1
})
[{'path': None, 'array': array([0.00140381, 0.0010376 , 0.00012207, ..., 0.00061035, 0.0005188 ,
       0.00039673], shape=(34304,)), 'sampling_rate': 16000}]
Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant'],
    num_rows: 1
})
[{'path': None, 'array': array([-0.00283813, -0.00460815, -0.00595093, ...,  0.00192261,
       -0.00036621, -0.00085449], shape=(28416,

In [ ]:
# Cell 3: Define Unified Data Collator
# دقیقاً مثل کدت (بدون تغییر)
@dataclass
class UnifiedDataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __post_init__(self):
        # Automatically set decoder_start_token_id from the tokenizer
        if hasattr(self.processor.tokenizer, "bos_token_id"):
            self.decoder_start_token_id = self.processor.tokenizer.bos_token_id
        elif hasattr(self.processor.tokenizer, "decoder_start_token_id"):
            self.decoder_start_token_id = self.processor.tokenizer.decoder_start_token_id
        else:
            raise ValueError("Processor's tokenizer must have 'bos_token_id' or 'decoder_start_token_id' attribute.")

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Step 1: Preprocess each feature to compute input_features and labels
        input_features_list = []
        label_features_list = []

        for feature in features:
            # Load audio data
            audio = feature.get("audio")
            if audio is None:
                raise ValueError("Feature missing 'audio' key.")

            # Compute log-Mel input features from input audio array (handles resampling if needed)
            input_features = self.processor.feature_extractor(
                audio["array"],
                sampling_rate=audio.get("sampling_rate", 16000),  # Default to 16000 if not provided
            ).input_features[0]

            input_features_list.append({"input_features": torch.tensor(input_features)})

            # Encode target text to label ids
            sentence = feature.get("sentence")
            if sentence is None:
                raise ValueError("Feature missing 'sentence' key.")
            labels = self.processor.tokenizer(sentence).input_ids
            label_features_list.append({"input_ids": labels})

        # Step 2: Pad the input features
        batch = self.processor.feature_extractor.pad(input_features_list, return_tensors="pt")

        # Step 3: Pad the label sequences
        labels_batch = self.processor.tokenizer.pad(label_features_list, return_tensors="pt")

        # Step 4: Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Step 5: If BOS token is appended at the beginning, remove it (as it's added later)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        # Step 6: Add labels to the batch
        batch["labels"] = labels

        return batch

print("✅ Data Collator defined!")


✅ Data Collator defined!


In [ ]:
# Cell 4: Define Custom PEFT Model Class
# دقیقاً مثل کدت (برای handle input_features در Whisper)
class WhisperPeftModel(PeftModelForSeq2SeqLM):
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        inputs_embeds=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        decoder_inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        task_ids=None,
        **kwargs
    ):
        # For Whisper, we expect 'input_features' in kwargs instead of 'input_ids'
        input_features = kwargs.pop("input_features", None)

        forward_kwargs = {}
        if input_features is not None:
            forward_kwargs["input_features"] = input_features
        if attention_mask is not None:
            forward_kwargs["attention_mask"] = attention_mask
        if inputs_embeds is not None:
            forward_kwargs["inputs_embeds"] = inputs_embeds
        if decoder_input_ids is not None:
            forward_kwargs["decoder_input_ids"] = decoder_input_ids
        if decoder_attention_mask is not None:
            forward_kwargs["decoder_attention_mask"] = decoder_attention_mask
        if decoder_inputs_embeds is not None:
            forward_kwargs["decoder_inputs_embeds"] = decoder_inputs_embeds
        if labels is not None:
            forward_kwargs["labels"] = labels
        if output_attentions is not None:
            forward_kwargs["output_attentions"] = output_attentions
        if output_hidden_states is not None:
            forward_kwargs["output_hidden_states"] = output_hidden_states
        if return_dict is not None:
            forward_kwargs["return_dict"] = return_dict

        # Pass any remaining kwargs
        forward_kwargs.update(kwargs)

        # Call the base model with adjusted kwargs
        return self.base_model(**forward_kwargs)

print("✅ Custom PEFT Model class defined!")


✅ Custom PEFT Model class defined!


In [ ]:
# Cell 5: Load Merged Model and Processor
# لود مدل merged (نه base openai) با quantization 4-bit
merged_model_path = "/data/whisper/train/medium/101-sergical/merged_model"  # path که در کد merge ذخیره کردی
compute_dtype = torch.bfloat16

# لود processor (از merged path یا base، اما از base می‌گیریم تا مطمئن باشیم)
processor = AutoProcessor.from_pretrained(merged_model_path)  # اگر processor در merged ذخیره شده، از اینجا لود کن؛ иначе از "openai/whisper-medium"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

print("Loading merged model in 4-bit... This may take a moment.")
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    merged_model_path,
    quantization_config=quantization_config,
    device_map="auto",
)

print("\n✅ Merged model loaded successfully in 4-bit!")
print("\nHere's the model structure:")
print(model)


Loading merged model in 4-bit... This may take a moment.

✅ Merged model loaded successfully in 4-bit!

Here's the model structure:
WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActiva

In [ ]:
# Cell 6: Prepare Model and Apply New LoRA
# اول prepare for k-bit training
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=32,
    lora_alpha=64,  # 2*r
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],  # normal: روی همه لایه‌ها اعمال می‌شه
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# اعمال LoRA جدید
model = get_peft_model(model, config)

# override با custom class
model.__class__ = WhisperPeftModel

model.print_trainable_parameters()  # چک کن params جدید (باید بیشتر باشه چون all layers)

print("✅ New LoRA applied to merged model!")


trainable params: 34,603,008 || all params: 798,460,928 || trainable%: 4.3337
✅ New LoRA applied to merged model!


In [ ]:
# Cell 7: Setup Metrics and Compute Function
# دقیقاً مثل کدت
output_dir = "/data/whisper/train/medium/102-sergical/continued_training"  # path جدید برای ذخیره checkpointهای جدید

wer_metric = evaluate.load("wer")

def normalize_persian_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('ي', 'ی').replace('ك', 'ک').replace('ء', '').replace('ؤ', 'و').replace('ئ', 'ی').replace('إ', 'ا')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str_normalized = [normalize_persian_text(text) for text in pred_str]
    label_str_normalized = [normalize_persian_text(text) for text in label_str]

    if not pred_str_normalized or not label_str_normalized:
        print("Warning: Empty predictions or labels in compute_metrics.")
        return {"wer": 1.0}  # Default to worst-case if empty

    wer = wer_metric.compute(predictions=pred_str_normalized, references=label_str_normalized)
    print(f"\n🎯 WER Score: {wer:.4f}")
    print(f"📊 Sample prediction: {pred_str_normalized[0][:150]}...")
    print(f"🎯 Sample reference: {label_str_normalized[0][:150]}...")
    return {"wer": wer}

# Small eval dataset
eval_dataset_small = dataset_dict["validation"].shuffle(seed=42).select(range(300))

print("✅ Metrics setup done!")



✅ Metrics setup done!


In [ ]:
# Cell 8: Training Arguments and Steps Calculation
# محاسبه steps و args (epochs رو به 6 تغییر دادم)
total_samples = len(dataset_dict["train"])
batch_size = 32
gradient_accumulation_steps = 2
effective_batch_size = batch_siz
e * gradient_accumulation_steps
steps_per_epoch = math.ceil(total_samples / effective_batch_size)  # Ceiling for complete coverage
total_epochs = 6
eval_every_n_epochs = 0.2
eval_steps = int(steps_per_epoch * eval_every_n_epochs)

print(f"Total samples: {total_samples}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Eval steps (every {eval_every_n_epochs} epochs): {eval_steps}")

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    report_to="none",
    lr_scheduler_type="cosine",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=5e-4,
    warmup_steps=int(0.1 * steps_per_epoch * total_epochs),
    num_train_epochs=total_epochs,
    eval_strategy="steps",
    eval_steps=eval_steps,
    fp16=False,
    bf16=True,
    per_device_eval_batch_size=batch_size,
    optim="adamw_torch",
    logging_steps=50,
    remove_unused_columns=False,
    label_names=["labels"],
    save_steps=eval_steps,
    save_total_limit=30,
    gradient_checkpointing=False,
    torch_compile=False,
    dataloader_pin_memory=False,
    predict_with_generate=True,
    generation_max_length=448,
)

print("✅ Training arguments set!")


Total samples: 262976
Steps per epoch: 4109
Eval steps (every 0.2 epochs): 821
✅ Training arguments set!


In [ ]:
# Cell 9: Initialize Trainer and Start Training
# initialize collator و trainer
data_collator = UnifiedDataCollatorSpeechSeq2SeqWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset_dict["train"],
    eval_dataset=eval_dataset_small,
    data_collator=data_collator,
    tokenizer=processor.tokenizer,  # Fixed: Use tokenizer for decoding
    compute_metrics=compute_metrics,
)

model.config.use_cache = False  # Silence warnings

print("\n🎬 Starting Training...")
print(f"📊 Train samples: {len(dataset_dict['train'])}")
print(f"📊 Eval samples: {len(eval_dataset_small)}")
print(f"⏱️ WER computed every {training_args.eval_steps} steps")

trainer.train()



🎬 Starting Training...
📊 Train samples: 262976
📊 Eval samples: 300
⏱️ WER computed every 821 steps


/tmp/ipykernel_43862/1474298772.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of 

Step,Training Loss,Validation Loss,Wer
821,0.074000,0.147355,0.293639


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161


🎯 WER Score: 0.2936
📊 Sample prediction: استعدادهای او دیر شکوفا شد...
🎯 Sample reference: استعدادهای او دیر شکوفا شد...


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
